# Exercice 1

---



In [4]:
import gymnasium as gym
import numpy as np

# Initialisation de l' environnement physique
env = gym.make('Humanoid-v5')
state ,info = env.reset()

print (" --- ANALYSE SYSTEME ---")
print ( f" Espace d'observation : {env.observation_space }")
print ( f" Espace d'action : {env.action_space }")
print ( f" Bornes des actions : Min{env.action_space.low[0]}, Max{env.action_space.high[0]}")

 --- ANALYSE SYSTEME ---
 Espace d'observation : Box(-inf, inf, (348,), float64)
 Espace d'action : Box(-0.4, 0.4, (17,), float32)
 Bornes des actions : Min-0.4000000059604645, Max0.4000000059604645


# Exercice 2

---



In [11]:
import gymnasium as gym          # Import de l'environnement RL
import numpy as np              # Import pour les calculs numériques

# Initialisation de l'environnement Humanoid (robot)
env = gym.make('Humanoid-v5')

# Nombre d'actions discrètes (macro-actions)
num_discrete_actions = 10

# Création de 10 macro-actions aléatoires (chaque action = 17 moteurs)
macro_actions = [env.action_space.sample() for _ in range(num_discrete_actions)]

# Q-table simplifiée : une valeur par action (on ignore l'état)
q_table = np.zeros(num_discrete_actions)

# Paramètres de l'apprentissage
alpha = 0.1     # Learning rate (vitesse d'apprentissage)
gamma = 0.99    # Facteur de réduction (importance du futur)
epsilon = 0.1   # Probabilité d'exploration

# Fonction epsilon-greedy pour choisir une action
def epsilon_greedy(q_values):
    # Générer un nombre aléatoire entre 0 et 1
    if np.random.uniform(0, 1) < epsilon:
        # Exploration : choisir une action aléatoire
        return np.random.randint(num_discrete_actions)
    # Exploitation : choisir la meilleure action connue
    return np.argmax(q_values)

# Nombre d'épisodes d'entraînement
episodes = 100

# Liste pour stocker les récompenses de chaque épisode
sarsa_rewards = []

# Boucle principale d'entraînement
for ep in range(episodes):

    # Réinitialisation de l'environnement (nouvel épisode)
    state, _ = env.reset()

    # Choix de la première action (SARSA = on-policy)
    action_idx = epsilon_greedy(q_table)

    # Initialisation de la récompense totale
    total_reward = 0

    # Variables de fin d'épisode
    done = False
    truncated = False

    # Boucle temporelle (jusqu'à chute ou fin)
    while not (done or truncated):

        # Récupérer la vraie action (vecteur des 17 moteurs)
        real_action = macro_actions[action_idx]

        # Appliquer l'action dans l'environnement
        next_state, reward, done, truncated, _ = env.step(real_action)

        # Choisir la prochaine action (SARSA = on-policy)
        next_action_idx = epsilon_greedy(q_table)

        # Mise à jour de la Q-table avec la formule SARSA
        q_table[action_idx] = q_table[action_idx] + alpha * (
            reward + gamma * q_table[next_action_idx] - q_table[action_idx]
        )

        # Passer à l'action suivante
        action_idx = next_action_idx

        # Ajouter la récompense obtenue
        total_reward += reward

    # Sauvegarder la récompense totale de l'épisode
    sarsa_rewards.append(total_reward)

# Fermer l'environnement après l'entraînement
env.close()

# Afficher les récompenses obtenues
print("Récompenses SARSA :", sarsa_rewards)

Récompenses SARSA : [np.float64(81.65269577195205), np.float64(73.69966761353021), np.float64(72.85632202239547), np.float64(76.07183516185269), np.float64(71.03685671240288), np.float64(73.67309534864731), np.float64(73.80420875072566), np.float64(72.06582854585722), np.float64(75.49495901489827), np.float64(75.0783426618378), np.float64(70.72317646811672), np.float64(74.33938564329189), np.float64(72.7375116670811), np.float64(83.04601988799982), np.float64(72.32922636632874), np.float64(74.71605482441237), np.float64(75.66832753959449), np.float64(72.96068392675464), np.float64(75.44964045969839), np.float64(70.27507789304863), np.float64(78.38113961031661), np.float64(68.78090639630679), np.float64(71.96768911196403), np.float64(77.61885235866198), np.float64(71.81671130197239), np.float64(75.283250054072), np.float64(76.5576507694906), np.float64(76.75087771020871), np.float64(76.75155896150096), np.float64(75.46699228605766), np.float64(72.12550424695878), np.float64(75.911561101

# Exercice 3

---




**Réponse à la question 1**:

Il est vital de punir les couples élevés car ils augmentent la consommation électrique, réduisent la durée de vie des moteurs et rendent le système inefficace énergétiquement.

+++++++++++++++++++++


**Réponse correcte**:

Sans coût de contact, l’agent peut exploiter la fonction de récompense en adoptant des comportements non physiques comme se laisser tomber pour avancer plus rapidement.

In [13]:
# Import de la classe PPO depuis Stable-Baselines3
from stable_baselines3 import PPO

# Import de la fonction pour évaluer le modèle
from stable_baselines3.common.evaluation import evaluate_policy

# Import de gymnasium pour créer l’environnement
import gymnasium as gym

# Création de l’environnement Humanoid (robot avec 17 moteurs et 348 états)
env = gym.make('Humanoid-v5')

# Création du modèle PPO
# "MlpPolicy" signifie qu'on utilise un réseau de neurones (Multi-Layer Perceptron)
model = PPO(
    "MlpPolicy",      # Type de réseau de neurones utilisé (entrée: état, sortie: action)
    env,              # Environnement sur lequel entraîner le modèle
    verbose=1,        # Affiche les informations pendant l'entraînement
    learning_rate=3e-4,  # Vitesse d’apprentissage (0.0003)
    batch_size=64     # Nombre d’exemples utilisés à chaque mise à jour
)

# Message pour indiquer le début de l'entraînement
print("Debut de l’entrainement PPO (Patientez ...)")

# Entraînement du modèle
# total_timesteps = nombre total d’interactions avec l’environnement
# Plus ce nombre est grand, plus le modèle apprend
model.learn(total_timesteps=150000)

# Sauvegarde du modèle entraîné dans un fichier
model.save("ppo_humanoid")

# Évaluation du modèle après entraînement
# On teste le modèle sur 5 épisodes pour mesurer sa performance
mean_reward, std_reward = evaluate_policy(
    model,                # Le modèle entraîné
    env,                  # L’environnement
    n_eval_episodes=5     # Nombre de tests
)

# Affichage de la performance moyenne et de la variation
print(f"Recompense moyenne PPO : {mean_reward:.2f} +/- {std_reward:.2f}")

# Fermeture de l’environnement pour libérer les ressources
env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Debut de l’entrainement PPO (Patientez ...)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.8     |
|    ep_rew_mean     | 97.5     |
| time/              |          |
|    fps             | 630      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 22.1        |
|    ep_rew_mean          | 97.8        |
| time/                   |             |
|    fps                  | 512         |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.018156497 |
|    clip_fraction        | 0.211       |
|    clip_range           | 0.2         |
|    entropy_loss         | -24.1       |
|    explained_variance   | 0.000554    |
|    learning_rate        | 0.

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Recompense moyenne PPO : 268.57 +/- 12.58


# Exercice 4

---



Dans un problème industriel comme l’optimisation de l’inclinaison des panneaux solaires, les variables d’entrée et de sortie sont continues et évoluent en temps réel. Les algorithmes classiques comme Q-Learning ou SARSA reposent sur une discrétisation des états et des actions, ce qui entraîne une perte d’information et une explosion combinatoire rendant leur utilisation impraticable. En revanche, une architecture Actor-Critic comme PPO utilise des réseaux de neurones pour modéliser directement une politique continue. Elle permet ainsi de généraliser à des situations nouvelles, de s’adapter dynamiquement aux changements de l’environnement, et d’optimiser efficacement le contrôle en temps réel. Cela en fait une solution beaucoup plus adaptée aux systèmes industriels complexes.